<a href="https://colab.research.google.com/github/atmakurinikitha2005-unqe/NLP-Preprocessing-Engine-Tasks/blob/main/Task2_NLP_Sentiment_Analysis_NLP_Pipeline%26ML_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Sentiment Analysis using NLP Pipeline & ML models**

In [1]:
# 1. INSTALL & IMPORT

!pip install nltk scikit-learn --quiet

import pandas as pd
import numpy as np
import re
import string

import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


# 2. UPLOAD DATASET

from google.colab import files
uploaded = files.upload()

file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name, encoding='latin-1')

print("Dataset Loaded")
print(df.head())


# 3. CHECK COLUMNS (IMPORTANT FIX)

print("\nColumns in dataset:", df.columns)

# Rename if needed (auto fix)
if 'review' not in df.columns:
    df.rename(columns={df.columns[0]: 'review'}, inplace=True)

if 'sentiment' not in df.columns:
    df.rename(columns={df.columns[1]: 'sentiment'}, inplace=True)

# Convert labels
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})


# 4. CLEANING FUNCTION (NO TOKENIZER ERROR)

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)

    words = text.split()

    words = [stemmer.stem(w) for w in words if w not in stop_words]

    return " ".join(words)

df['cleaned'] = df['review'].apply(clean_text)

print("\n Cleaned Sample:")
print(df[['review', 'cleaned']].head())


# 5. FEATURE ENGINEERING

bow = CountVectorizer(max_features=3000)
tfidf = TfidfVectorizer(max_features=3000)

X_bow = bow.fit_transform(df['cleaned'])
X_tfidf = tfidf.fit_transform(df['cleaned'])

y = df['sentiment']


# 6. SPLIT

X_train_bow, X_test_bow, y_train, y_test = train_test_split(X_bow, y, test_size=0.2, random_state=42)
X_train_tfidf, X_test_tfidf, _, _ = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)


# 7. MODEL FUNCTION

def run_model(model, X_train, X_test):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    print("\nModel:", model.__class__.__name__)
    print("Accuracy:", accuracy_score(y_test, pred))
    print("Precision:", precision_score(y_test, pred))
    print("Recall:", recall_score(y_test, pred))
    print("F1:", f1_score(y_test, pred))


# 8. TRAIN MODELS (BoW)

print("\n===== BoW =====")

run_model(LogisticRegression(max_iter=200), X_train_bow, X_test_bow)
run_model(MultinomialNB(), X_train_bow, X_test_bow)
run_model(DecisionTreeClassifier(), X_train_bow, X_test_bow)


# 9. TRAIN MODELS (TF-IDF)

print("\n===== TF-IDF =====")

run_model(LogisticRegression(max_iter=200), X_train_tfidf, X_test_tfidf)
run_model(MultinomialNB(), X_train_tfidf, X_test_tfidf)
run_model(DecisionTreeClassifier(), X_train_tfidf, X_test_tfidf)


# 10. INSIGHTS

print("\nDONE")
print("TF-IDF + Logistic Regression usually best")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Saving IMDB Dataset.csv to IMDB Dataset.csv
Dataset Loaded
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive

Columns in dataset: Index(['review', 'sentiment'], dtype='object')

 Cleaned Sample:
                                              review  \
0  One of the other reviewers has mentioned that ...   
1  A wonderful little production. <br /><br />The...   
2  I thought this was a wonderful way to spend ti...   
3  Basically there's a family where a little boy ...   
4  Petter Mattei's "Love in the Time of Money" is...   

                                             cleaned  
0  one review mention watch oz episod hook right ...  
1  wonder littl product br br 